In [ ]:
#Set up paths
import sys
import os

from Utils import utils as uti

from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import MyConstants as Co


import regrid_wavecube as rwv


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some other useful packages 
import importlib
import copy
import time
import cftime
import yaml
import glob
#from box import Box #???

importlib.reload( rwv )
importlib.reload(MkP)
importlib.reload(GrU)

pi_ = Co.pi()
R_earth = Co.Rearth()

print( R_earth )

In [ ]:
import glob
import os

pattern = '/glade/derecho/scratch/juliob/archive//c153_topfix_ne240pg3_FMTHIST_xic_x02/atm/WaveCube/c153_topfix_ne240pg3_FMTHIST_xic_x02.cam.h1i.2004-07-*.nc'
files = sorted(glob.glob(pattern))

print(f"Found {len(files)} files")



In [ ]:
#print( files )
X=xr.open_dataset( files[0] )
print( X.sizes['lat'] )
print( X.lat[300].values )
nt,nz,ny,nx= len(files),X.sizes['lev'],X.sizes['lat'],X.sizes['lon'] 
print( nt,nz,ny,nx )
wp_tyx=np.zeros( (nt,ny,nx ) )
up_tyx=np.zeros( (nt,ny,nx ) )
wp_tzx=np.zeros( (nt,nz,nx ) )

tbk_tzx = np.zeros( (nt,nz,nx ) )
ubk_tzx = np.zeros( (nt,nz,nx ) )


In [ ]:
n=0
lat_sel = -52.4
for f in files:
    X=xr.open_mfdataset(f , data_vars='different', coords='different' , compat='no_conflicts' )
    wp_ =X.wp.sel(lev=1., method='nearest').values
    wp_tyx[n,:,:] = wp_
    up_ =X.up.sel(lev=1.,method='nearest').values
    up_tyx[n,:,:] = up_

    n=n+1
    if (n%10==0):
        print( f )

In [ ]:
upwp_tyx = up_tyx * wp_tyx
plt.contourf( upwp_tyx[:,300,:] ,levels=np.linspace(-100,100,num=50 ) ,cmap='bwr' )
plt.colorbar()

In [ ]:
#####################################
# Initialize regrid-object library
RgObLib={}
RgOb_latlonOxO_x_fv1x1  = GrU.regrid_object_lib(RgOb=RgObLib, src='latlonOxO', dst='fv1x1',   RegridMethod='CONSERVE_2ND')


In [ ]:
import RegridField as RgF



upwp_tyx_x1R=RgF.Horz(xfld_Src=upwp_tyx , Src='latlonOxO', Dst='fv1x1' , RegridObj_In= RgOb_latlonOxO_x_fv1x1   ) 

lat1R,lon1R = GrU.latlon( grid='fv1x1' )




In [ ]:
lat,lon=X.lat.values,X.lon.values
print( lat[300] )
print( lat1R[40] )

In [ ]:
plt.contourf( upwp_tyx_x1R[:,40,:] ,levels=np.linspace(-100,100,num=50 ) ,cmap='bwr' )
plt.colorbar()

In [ ]:

plev=X.lev.values
lon=X.lon.values
zlev= -7_000.*np.log( plev/1000. )
print( plev[-1] )

In [ ]:
fig,ax=plt.subplots( 1,1 , figsize=(40,6) )
c=ax.contourf( lon,zlev,wp_tzx[13,:,:] , cmap='bwr' )
plt.colorbar( c )

In [ ]:
from Regridder import VertRegridFlexLL as vrg

#zSrc=np.zeros( (nt,nz,nx ) )

nz_m=128
zlev_m = np.linspace( 0., zlev[0], num=nz_m )

zSrc = zlev[None, :, None]
zSrc = np.broadcast_to(zlev[None, :, None], (nt, nz, nx))
zDst = zlev_m[None, :, None]
zDst = np.broadcast_to(zlev_m[None, :, None], (nt, nz_m, nx))

#plt.contourf( zSrc[15,:,:] )
#plt.colorbar()

wp_tzx_m = vrg.VertRG( a_x=wp_tzx , zSrc=zSrc, zDst=zDst, Gridkey='tzc' , fill_value='extrapolate', kind='linear' )
ubk_tzx_m = vrg.VertRG( a_x=ubk_tzx , zSrc=zSrc, zDst=zDst, Gridkey='tzc' , fill_value='extrapolate', kind='linear' )
tbk_tzx_m = vrg.VertRG( a_x=tbk_tzx , zSrc=zSrc, zDst=zDst, Gridkey='tzc' , fill_value='extrapolate', kind='linear' )




In [ ]:
fig,ax=plt.subplots( 1,1 , figsize=(40,6) )
c=ax.contourf( lon,zlev_m,wp_tzx_m[12,:,:] , cmap='bwr' ,levels=np.linspace(-9.,9., 10 ))
l=ax.contour( lon,zlev_m,ubk_tzx_m[12,:,:] , levels=np.linspace(-100.,100., 11 ) )

#ax.set_xlim(0,100)
plt.colorbar( c )

In [ ]:
#print(zlev_m[0:5]-zlev_m[1:6])
dz=zlev_m[1]-zlev_m[0]
drlon = (pi_ / 180. ) * (lon[1]-lon[0])
rlat  = (pi_ / 180. ) * lat_sel
dx= np.cos( rlat ) * R_earth * drlon
dt=6.*3_600.

print( dt, dz, dx )

In [ ]:
%%time
from scipy.fft import fftn, fftfreq
from scipy.signal.windows import tukey


# 1) (optional) remove means

wp_4_fft =wp_tzx_m[:,80,0:600]

#nt_fft, nz_fft, nx_fft = wp_4_fft.shape
nt_fft , nx_fft = wp_4_fft.shape

print( nt_fft , nx_fft )

wt = tukey(nt_fft,  alpha=0.2)[:, None]
#wz = tukey(nz,  alpha=0.2)[None, :, None]
wx = tukey(nx_fft,  alpha=0.2)[None, :]


wp0 = wp_4_fft - wp_4_fft.mean(axis=0, keepdims=True)      # remove time-mean
wp0 = wp0 - wp0.mean(axis=1, keepdims=True)    # remove zonal-mean


wpw = wp0 * wt * wx



# 2) 3D complex FFT over (t,z,x)
W = fftn(wpw, axes=(0, 1))                  # -> (nt, nz, nx)
P = np.abs(W)**2                                # power in (f,m,k) bins

# 3) physical axes (cycles units)
f = fftfreq(nt_fft, d=dt)                          # Hz (cycles/s), signed
m = fftfreq(nz, d=dz)                          # cycles/m, signed (vertical wavenumber)
k = fftfreq(nx_fft, d=dx)                          # cycles/m, signed (zonal wavenumber)

F, M, K = np.meshgrid(f, m, k, indexing="ij")


In [ ]:
plt.contour(wpw**2)

In [ ]:
print( zlev_m[80] )
print(P.shape)
print(f.shape)
print(k.shape)
print(m.shape)


In [ ]:
kk=0.5*1e-5
print( (1./kk) / 1000. )

In [ ]:
plt.contour( k,f,P )

In [ ]:
P_fk = P.sum(axis=1)                            # collapse over m -> (f,k)

F2, K2 = np.meshgrid(f, k, indexing="ij")
c = np.where(K2 != 0.0, F2 / K2, np.nan)         # m/s

mask = np.isfinite(c) & (np.abs(K2) > 0.0)
c_bins = np.linspace(-200, 200, 401)             # choose range/resolution
Pc_east = np.zeros(len(c_bins)-1)
Pc_west = np.zeros(len(c_bins)-1)

c_flat = c[mask]
p_flat = P_fk[mask]
k_flat = K2[mask]

inds = np.digitize(c_flat, c_bins) - 1
ok = (inds >= 0) & (inds < len(Pc_east))

np.add.at(Pc_east, inds[ok & (k_flat > 0)], p_flat[ok & (k_flat > 0)])
np.add.at(Pc_west, inds[ok & (k_flat < 0)], p_flat[ok & (k_flat < 0)])

c_centers = 0.5*(c_bins[:-1] + c_bins[1:])


In [ ]:
P_m = P.sum(axis=(0, 2))                         # -> (m,)
k_pos = (k > 0)
k_neg = (k < 0)

P_m_east = P[:, :, k_pos].sum(axis=(0, 2))       # -> (m,)
P_m_west = P[:, :, k_neg].sum(axis=(0, 2))       # -> (m,)
